# Credit Card Fraud Detection using Machine Learning

In this notebook, we build an effective fraud detection model on highly unbalanced transaction data. We apply **Random Under-Sampling** to balance normal and fraudulent transactions, followed by **Logistic Regression**.


### Importing the Dependencies


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


### Data Collection and Preprocessing


In [ ]:
# Loading the dataset to a Pandas DataFrame
# Generating representative transactions for reproducible demonstration
np.random.seed(42)
n_legit = 492 * 5
n_fraud = 492

# Features V1-V28 plus Amount
V_legit = np.random.normal(0, 1, size=(n_legit, 28))
V_fraud = np.random.normal(1.2, 1.5, size=(n_fraud, 28))

Amt_legit = np.random.exponential(88, size=(n_legit, 1))
Amt_fraud = np.random.exponential(122, size=(n_fraud, 1))

Class_legit = np.zeros((n_legit, 1))
Class_fraud = np.ones((n_fraud, 1))

legit_data = np.hstack([V_legit, Amt_legit, Class_legit])
fraud_data = np.hstack([V_fraud, Amt_fraud, Class_fraud])

columns = [f'V{i}' for i in range(1, 29)] + ['Amount', 'Class']
credit_card_data = pd.DataFrame(np.vstack([legit_data, fraud_data]), columns=columns)
credit_card_data['Class'] = credit_card_data['Class'].astype(int)
credit_card_data.head()


In [ ]:
# First and last rows
print("Dataset Shape:", credit_card_data.shape)
credit_card_data['Class'].value_counts()


### This Dataset is Highly Unbalanced:
* **0 --> Normal Transaction**
* **1 --> Fraudulent Transaction**


### Data Visualization


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='Class', data=credit_card_data, palette=['#2ecc71', '#e74c3c'])
plt.title('Imbalanced Transaction Distribution')
plt.xticks(ticks=[0, 1], labels=['Normal (0)', 'Fraud (1)'])
plt.show()


### Handling Imbalanced Data: Random Under-Sampling


In [ ]:
legit = credit_card_data[credit_card_data.Class == 0]
fraud = credit_card_data[credit_card_data.Class == 1]

# Statistical measures of transaction amounts
print("Legit Amount:")
print(legit.Amount.describe())
print("\nFraud Amount:")
print(fraud.Amount.describe())

# Under-sample normal transactions to match fraud count
legit_sample = legit.sample(n=len(fraud), random_state=2)
new_dataset = pd.concat([legit_sample, fraud], axis=0)
print("\nBalanced Dataset Class Counts:")
print(new_dataset['Class'].value_counts())


### Splitting the Data into Features & Targets


In [ ]:
X = new_dataset.drop(columns='Class', axis=1)
Y = new_dataset['Class']
print(X.shape, Y.shape)


### Splitting into Training and Testing Data


In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=2)
print("Train:", X_train.shape, "Test:", X_test.shape)


### Model Training --> Logistic Regression


In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, Y_train)


### Model Evaluation


In [ ]:
# Accuracy on training data
X_train_pred = model.predict(X_train)
print('Accuracy on Training data : ', round(accuracy_score(Y_train, X_train_pred) * 100, 2), '%')

# Accuracy on test data
X_test_pred = model.predict(X_test)
print('Accuracy on Test data     : ', round(accuracy_score(Y_test, X_test_pred) * 100, 2), '%')


### Building a Predictive System


In [ ]:
sample_transaction = X_test.iloc[0].values

# Reshape for single prediction
sample_reshaped = sample_transaction.reshape(1, -1)
prediction = model.predict(sample_reshaped)
print("Prediction output:", prediction)

if prediction[0] == 0:
    print('>>> Result: Normal Transaction (Approved)')
else:
    print('>>> Result: FRAUDULENT TRANSACTION (Alert / Blocked)')
